# 01 — Understanding C-MAPSS FD001 Sensor / Time-Series Data

This notebook documents the **sensor/time-series structure**, performs **EDA**, and records **cleaning diagnostics** for NASA C-MAPSS FD001 Remaining Useful Life (RUL) prediction.

## What each row means
- One row = one operational **cycle** for one turbofan **engine** (`unit_id`).
- Training engines run until failure (full degradation trajectories).
- Test engines are truncated before failure; true RUL at the last cycle is in `RUL_FD001.txt`.


In [ ]:
import sys
from pathlib import Path
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

from cmapss.download import download_fd001
from cmapss.load import load_fd001, describe_schema
from cmapss.cleaning import cleaning_diagnostics
from cmapss.features import add_rul_labels
from cmapss.config import SENSOR_COLS, SETTING_COLS

sns.set_theme(style='whitegrid', context='notebook')
download_fd001()
train, test, test_rul = load_fd001()
print(describe_schema())
print('Train shape:', train.shape, '| Test shape:', test.shape, '| Test RUL engines:', len(test_rul))
train.head()


## Train vs test structure
Training trajectories are longer on average because they continue to failure. Test trajectories stop earlier — that is why we need a separate RUL label file.


In [ ]:
train_len = train.groupby('unit_id')['cycle'].max()
test_len = test.groupby('unit_id')['cycle'].max()
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].hist(train_len, bins=20, color='#4C72B0', edgecolor='black')
ax[0].set_title('Train: cycles to failure per engine')
ax[0].set_xlabel('Max cycle')
ax[1].hist(test_len, bins=20, color='#55A868', edgecolor='black')
ax[1].set_title('Test: observed cycles per engine')
ax[1].set_xlabel('Max cycle')
plt.tight_layout()
plt.show()
print(train_len.describe())
print(test_len.describe())


## Cleaning diagnostics
FD001 has a single operating condition, so some settings/sensors are constant and carry no information. We drop them to reduce noise.


In [ ]:
report = cleaning_diagnostics(train)
print('Missing values:', report.n_missing)
print('Constant sensors:', report.constant_sensors)
print('Near-constant sensors:', report.near_constant_sensors)
print('Dropped columns:', report.dropped_columns)
print('Kept sensors:', report.kept_sensors)
print('Kept settings:', report.kept_settings)


## RUL target distribution
We compute uncapped RUL as `max_cycle - cycle`. For training we use a piecewise cap of 125 cycles (standard C-MAPSS practice) so early healthy operation is not over-emphasized.


In [ ]:
train_rul = add_rul_labels(train, cap=125)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].hist(train_rul['RUL_uncapped'], bins=40, color='#4C72B0', edgecolor='black')
ax[0].set_title('Uncapped RUL')
ax[1].hist(train_rul['RUL'], bins=40, color='#C44E52', edgecolor='black')
ax[1].set_title('Capped RUL (cap=125)')
plt.tight_layout()
plt.show()


## Example engine degradation trajectories
Selected sensors typically show clear drift as the engine approaches failure.


In [ ]:
sample_units = [1, 2, 3]
sensors_to_plot = ['sensor_4', 'sensor_11', 'sensor_15']
fig, axes = plt.subplots(len(sensors_to_plot), 1, figsize=(10, 8), sharex=True)
for ax, sensor in zip(axes, sensors_to_plot):
    for u in sample_units:
        d = train[train['unit_id'] == u]
        ax.plot(d['cycle'], d[sensor], label=f'engine {u}')
    ax.set_ylabel(sensor)
    ax.legend(loc='best', fontsize=8)
axes[-1].set_xlabel('Cycle')
axes[0].set_title('Sensor trajectories for sample engines')
plt.tight_layout()
plt.show()


## Correlation among kept sensors


In [ ]:
kept = report.kept_sensors
corr = train[kept].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap='coolwarm', center=0, square=True)
plt.title('Correlation heatmap (kept sensors)')
plt.tight_layout()
plt.show()


## EDA takeaways
1. FD001 is multivariate **time series per engine**, not i.i.d. rows.
2. Several sensors/settings are constant under the single operating condition — drop them.
3. Sensor trends near end-of-life motivate **rolling window features**.
4. Because consecutive rows belong to the same engine, validation must be **engine-wise** to avoid leakage.
